In [ ]:
# =====================================================================
# CELL 1: INSTALASI, IMPORT & SETUP GROQ API (AMAN VIA SECRETS)
# =====================================================================
!pip install groq gradio pysastrawi numpy -q

from groq import Groq
import gradio as gr
import joblib
import re
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from google.colab import userdata  # Import untuk mengambil Secret

# Download data NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# ===== AMBIL API KEY DARI COLAB SECRETS =====
# Pastikan nama di panel Secrets adalah 'GROQ_API_KEY' dan toggle 'Notebook access' sudah ON (biru)
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Cek apakah API Key berhasil diambil
if not GROQ_API_KEY:
    raise ValueError("⚠️ API Key tidak ditemukan! Pastikan: 1) Nama secret di panel Secrets adalah 'GROQ_API_KEY', 2) Toggle 'Notebook access' sudah dinyalakan (ON).")

# Inisialisasi Groq Client
client_groq = Groq(api_key=GROQ_API_KEY)

print("✅ Setup Groq API berhasil! API Key aman terbaca dari Secrets.")
print("🚀 Model yang digunakan: LLaMA 3.3 70B (via Groq LPU)")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


✅ Setup Groq API berhasil! API Key aman terbaca dari Secrets.
🚀 Model yang digunakan: LLaMA 3.3 70B (via Groq LPU)


In [ ]:
from google.colab import files

# =====================================================================
# CELL 2: UPLOAD MODEL SVM
# =====================================================================
print("📂 Upload file model_svm_best.pkl dan tfidf_vectorizer.pkl:")
uploaded = files.upload()

model_svm = joblib.load('model_svm_best.pkl')
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
print("✅ Model berhasil dimuat!")

📂 Upload file model_svm_best.pkl dan tfidf_vectorizer.pkl:


Saving 1_Training_Model.ipynb to 1_Training_Model.ipynb
Saving 3_AI_Insight_Gradio_Groq.ipynb to 3_AI_Insight_Gradio_Groq.ipynb
✅ Model berhasil dimuat!


In [ ]:
# =====================================================================
# CELL 3: FUNGSI PREPROCESSING (DENGAN KAMUS LENGKAP)
# =====================================================================
KAMUS_NORMALISASI = {
    # === KATA NEGASI ===
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'enggak': 'tidak',
    'ndak': 'tidak', 'gk': 'tidak', 'tdk': 'tidak',
    'bukan': 'bukan', 'belom': 'belum', 'blm': 'belum',

    # === KATA GANTI & PARTIKEL ===
    'udah': 'sudah', 'sdh': 'sudah', 'dah': 'sudah', 'deh': 'sudah',
    'syg': 'sayang', 'aq': 'saya', 'aku': 'saya', 'gue': 'saya', 'gw': 'saya',
    'elo': 'kamu', 'lu': 'kamu', 'km': 'kamu', 'anda': 'anda',
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'thd': 'terhadap',
    'dr': 'dari', 'krn': 'karena', 'karna': 'karena', 'spy': 'supaya',
    'biar': 'agar', 'klo': 'kalau', 'klu': 'kalau', 'kalo': 'kalau', 'kl': 'kalau',
    'dlm': 'dalam', 'ttg': 'tentang', 'ttng': 'tentang', 'slh': 'salah',
    'bgt': 'sangat', 'banget': 'sangat', 'sekali': 'sangat', 'bnget': 'sangat',
    'jg': 'juga', 'uga': 'juga', 'juga': 'juga',
    'aja': 'saja', 'doang': 'saja', 'cuma': 'hanya', 'cm': 'hanya',
    'kok': 'kok', 'sih': 'sih', 'dong': 'dong', 'nih': 'ini', 'tu': 'itu',

    # === KATA SIFAT POSITIF ===
    'mantap': 'bagus', 'mantul': 'bagus', 'keren': 'bagus', 'good': 'bagus',
    'nice': 'bagus', 'puas': 'puas', 'senang': 'senang', 'top': 'bagus',
    'jos': 'bagus', 'mantaap': 'bagus', 'sip': 'bagus', 'siip': 'bagus',
    'ok': 'baik', 'oke': 'baik', 'okelah': 'baik', 'okk': 'baik',
    'recommended': 'bagus', 'rekomen': 'bagus', 'recomended': 'bagus',
    'memuaskan': 'puas', 'membantu': 'bantu', 'helpful': 'bantu',
    'useful': 'guna', 'berguna': 'guna', 'praktis': 'praktis',
    'efisien': 'efisien', 'efektif': 'efektif', 'mudah': 'mudah',
    'gampang': 'mudah', 'simple': 'sederhana', 'simpel': 'sederhana',
    'cepat': 'cepat', 'quick': 'cepat', 'fast': 'cepat',
    'lancar': 'lancar', 'smooth': 'lancar', 'nyaman': 'nyaman',
    'aman': 'aman', 'safe': 'aman', 'terpercaya': 'percaya',

    # === KATA SIFAT NEGATIF ===
    'lemot': 'lambat', 'lelet': 'lambat', 'lamban': 'lambat', 'slow': 'lambat',
    'ribet': 'rumit', 'susah': 'sulit', 'sulit': 'sulit', 'rumit': 'rumit',
    'gagal': 'gagal', 'fail': 'gagal', 'failed': 'gagal',
    'error': 'error', 'err': 'error', 'eror': 'error',
    'rusak': 'rusak', 'broken': 'rusak', 'damage': 'rusak',
    'jelek': 'jelek', 'buruk': 'buruk', 'bad': 'buruk', 'poor': 'buruk',
    'parah': 'parah', 'terrible': 'parah', 'horrible': 'parah',
    'kecewa': 'kecewa', 'disappointed': 'kecewa', 'disappointing': 'kecewa',
    'marah': 'marah', 'angry': 'marah', 'frustrated': 'frustrasi',
    'kesal': 'kesal', 'sebel': 'kesal', 'annoyed': 'kesal',
    'ganggu': 'ganggu', 'mengganggu': 'ganggu', 'annoying': 'ganggu',
    'crash': 'error', 'force close': 'error', 'fc': 'error',
    'ngelag': 'lambat', 'lag': 'lambat', 'laggy': 'lambat',
    'hang': 'error', 'freeze': 'error', 'not responding': 'error',
    'bug': 'error', 'buggy': 'error', 'glitch': 'error',
    'confusing': 'bingung', 'membingungkan': 'bingung', 'bingung': 'bingung',
    'complicated': 'rumit', 'complex': 'rumit',

    # === ISTILAH APLIKASI & LAYANAN ===
    'aplikasi': 'aplikasi', 'app': 'aplikasi', 'apps': 'aplikasi',
    'sim': 'sim', 'stnk': 'stnk', 'skck': 'skck', 'etle': 'etle',
    'tilang': 'tilang', 'e-tilang': 'tilang', 'etilang': 'tilang',
    'polri': 'polri', 'polisi': 'polisi', 'kepolisian': 'polisi',
    'pengaduan': 'aduan', 'adu': 'aduan', 'complain': 'aduan',
    'complaint': 'aduan', 'lapor': 'lapor', 'report': 'lapor',
    'layanan': 'layanan', 'service': 'layanan', 'pelayanan': 'layanan',
    'fitur': 'fitur', 'feature': 'fitur', 'menu': 'menu',
    'tampilan': 'tampilan', 'display': 'tampilan', 'ui': 'tampilan',

    # === KATA KERJA UMUM ===
    'bisa': 'bisa', 'bsa': 'bisa', 'dpt': 'dapat', 'dapet': 'dapat',
    'krja': 'kerja', 'bntu': 'bantu', 'mbantu': 'bantu', 'membantu': 'bantu',
    'memudahkan': 'mudah', 'bantu': 'bantu', 'help': 'bantu',
    'gunakan': 'guna', 'pakai': 'guna', 'use': 'guna', 'using': 'guna',
    'download': 'unduh', 'unduh': 'unduh', 'install': 'instal',
    'update': 'perbarui', 'perbarui': 'perbarui', 'upgrade': 'perbarui',
    'login': 'masuk', 'masuk': 'masuk', 'sign in': 'masuk',
    'logout': 'keluar', 'keluar': 'keluar', 'sign out': 'keluar',
    'register': 'daftar', 'daftar': 'daftar', 'sign up': 'daftar',
    'verifikasi': 'verifikasi', 'verify': 'verifikasi',
    'upload': 'unggah', 'unggah': 'unggah', 'submit': 'kirim',
    'kirim': 'kirim', 'send': 'kirim', 'loading': 'muat',
    'muat': 'muat', 'process': 'proses', 'proses': 'proses',
}

KATA_NEGASI = {'tidak', 'tak', 'bukan', 'belum', 'tanpa', 'jangan', 'tiada', 'kurang'}
stopwords_id = set(stopwords.words('indonesian')) - KATA_NEGASI
stemmer = StemmerFactory().create_stemmer()

def preprocessing(teks):
    if not isinstance(teks, str): return ""
    teks = teks.lower()
    teks = re.sub(r'http\S+|www\S+|@\w+|#\w+|\d+|[^\w\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()

    kata = [KAMUS_NORMALISASI.get(k, k) for k in teks.split()]
    tokens = word_tokenize(' '.join(kata))
    tokens = [t for t in tokens if t not in stopwords_id]

    tokens_stemmed = [t if t in KATA_NEGASI else stemmer.stem(t) for t in tokens]
    return ' '.join(tokens_stemmed)

print(f"✅ Fungsi preprocessing siap! (Kamus berisi {len(KAMUS_NORMALISASI)} kata)")

✅ Fungsi preprocessing siap! (Kamus berisi 217 kata)


In [ ]:
# =====================================================================
# CELL 4: FUNGSI AI INSIGHT KOMPREHENSIF (GROQ - LLAMA 3.3 70B)
# =====================================================================
def generate_ai_insight(teks_asli, prediksi, confidence):
    """
    Memanggil Groq API (LLaMA 3.3 70B) untuk menghasilkan analisis mendalam
    berdasarkan ulasan dan hasil prediksi SVM
    """
    prompt = f"""Anda adalah analis sentimen senior dan konsultan UX/UI profesional yang bertugas menganalisis ulasan pengguna aplikasi Super App Presisi Polri.

ULASAN PENGGUNA:
"{teks_asli}"

HASIL KLASIFIKASI MODEL SVM:
- Sentimen: {prediksi.upper()}
- Tingkat Keyakinan: {confidence:.2f}%

TUGAS ANDA:
Berikan analisis KOMPREHENSIF dan DETAIL dalam format berikut:

📝 **1. Analisis Konteks & Makna:**
[Jelaskan makna tersirat dari ulasan ini secara mendalam. Apa yang sebenarnya ingin disampaikan pengguna? Apa konteks di balik ulasan tersebut?]

🎯 **2. Aspek Layanan yang Dibahas:**
[Identifikasi secara spesifik fitur/layanan apa yang dibahas dalam ulasan ini. Pilih dari: SIM, STNK, SKCK, ETLE/Tilang Elektronik, Pengaduan Masyarakat, SP2HP, Performa Aplikasi, UI/UX, atau Fitur Lainnya. Jika ada beberapa aspek, sebutkan semua.]

😊 **3. Analisis Emosi Pengguna:**
[Analisis emosi yang dirasakan pengguna. Pilih emosi dominan dari: Senang/Puas, Frustrasi/Marah, Bingung, Kecewa, Cemas, atau Netral. Jelaskan mengapa emosi tersebut muncul.]

🔍 **4. Identifikasi Masalah (Jika Ada):**
[Jika ulasan mengandung keluhan, identifikasi jenis masalah secara spesifik: Bug/Error, Performa Lambat, UI Membingungkan, Fitur Tidak Berfungsi, Proses Rumit, atau Lainnya. Jika tidak ada masalah, tulis "Tidak ada masalah spesifik teridentifikasi."]

💡 **5. Rekomendasi untuk Pengembang (Actionable):**
[Berikan 2-3 rekomendasi KONKRET dan SPESIFIK yang harus dilakukan pengembang Super App Presisi Polri. Rekomendasi harus actionable dan terukur. Contoh: "Tambahkan indikator loading saat proses upload foto SKCK" atau "Perbaiki validasi input pada form perpanjangan SIM."]

⚠️ **6. Tingkat Prioritas & Justifikasi:**
[Berikan angka 1-5 dengan justifikasi detail:
- 1 = Rendah (pujian rutin, tidak perlu tindakan segera)
- 2 = Sedang (masukan perbaikan, bisa dijadwalkan)
- 3 = Cukup Penting (masalah minor, perlu ditangani dalam sprint berikutnya)
- 4 = Penting (masalah signifikan, perlu prioritas tinggi)
- 5 = Sangat Mendesak (bug kritis, banyak pengguna terdampak, perlu hotfix)]

📊 **7. Dampak terhadap Pengguna:**
[Jelaskan seberapa besar dampak masalah/fitur ini terhadap pengalaman pengguna secara keseluruhan. Apakah ini mempengaruhi banyak pengguna atau hanya kasus individual?]

🔗 **8. Informasi Tambahan (Opsional):**
[Berikan konteks tambahan jika relevan, misalnya: perbandingan dengan aplikasi serupa, best practice dari aplikasi layanan publik lain, atau tren industri yang terkait.]

ATURAN PENTING:
- Gunakan bahasa Indonesia yang profesional namun mudah dipahami
- Berikan analisis yang objektif dan berbasis data
- Hindari asumsi yang tidak berdasar
- Jika ulasan terlalu singkat atau ambigu, akui keterbatasan analisis
- Format menggunakan markdown yang rapi dan mudah dibaca
- Total panjang respons: 300-500 kata"""

    try:
        # Panggil Groq API
        completion = client_groq.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": """Anda adalah analis sentimen senior dan konsultan UX/UI profesional dengan pengalaman 10+ tahun dalam menganalisis umpan balik pengguna aplikasi layanan publik.

Tugas Anda adalah memberikan analisis yang:
1. MENDALAM - tidak hanya permukaan, tetapi menggali makna tersirat
2. SPESIFIK - menyebutkan aspek, fitur, dan masalah secara detail
3. ACTIONABLE - rekomendasi yang bisa langsung ditindaklanjuti
4. OBJEKTIF - berbasis data dan fakta, bukan asumsi
5. KOMPREHENSIF - mencakup berbagai dimensi analisis

Anda memiliki pengetahuan mendalam tentang:
- Aplikasi Super App Presisi Polri dan fitur-fiturnya
- Best practice UX/UI untuk aplikasi layanan publik
- Pola keluhan umum pengguna aplikasi pemerintah
- Metrik evaluasi kualitas aplikasi mobile

Berikan respons dalam bahasa Indonesia yang profesional, terstruktur, dan mudah dipahami."""
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=1000,  # Diperbesar agar respons lebih detail
            top_p=0.9
        )

        return completion.choices[0].message.content

    except Exception as e:
        return f"""⚠️ **Gagal menghasilkan insight AI**

Error: {str(e)}

**Kemungkinan penyebab:**
- API key Groq tidak valid atau sudah kedaluwarsa
- Quota request sudah habis (limit: 30 request/menit)
- Koneksi internet terputus

**Solusi:**
1. Periksa API key Anda di https://console.groq.com
2. Tunggu 1 menit jika quota habis
3. Pastikan koneksi internet stabil

Silakan coba lagi dalam beberapa saat."""

print("✅ Fungsi AI Insight Komprehensif siap!")
print("🧠 Model akan memberikan analisis mendalam dengan 8 dimensi analisis")

✅ Fungsi AI Insight Komprehensif siap!
🧠 Model akan memberikan analisis mendalam dengan 8 dimensi analisis


In [ ]:
# =====================================================================
# CELL 5: FUNGSI PREDIKSI + AI INSIGHT
# =====================================================================
import numpy as np

def prediksi_dengan_insight(teks_ulasan):
    """Fungsi utama yang menggabungkan prediksi SVM + AI Insight dari Groq"""
    if not teks_ulasan or len(teks_ulasan.strip()) == 0:
        return "⚠️ Silakan masukkan teks ulasan terlebih dahulu!"

    # 1. Preprocessing
    teks_bersih = preprocessing(teks_ulasan)

    # 2. TF-IDF Transform
    vektor = tfidf_vectorizer.transform([teks_bersih])

    # 3. Prediksi SVM
    pred = model_svm.predict(vektor)[0]

    # 4. Hitung confidence (gunakan decision_function)
    distances = model_svm.decision_function(vektor)[0]
    max_distance = max(abs(distances))
    confidence = 1 / (1 + np.exp(-max_distance)) * 100

    # 5. Emoji mapping
    emoji_map = {'positif': '😊 POSITIF', 'netral': '😐 NETRAL', 'negatif': '😠 NEGATIF'}

    # 6. Generate AI Insight via Groq
    insight = generate_ai_insight(teks_ulasan, pred, confidence)

    # 7. Format output
    hasil = f"""# 🚔 Hasil Analisis Sentimen Super App Presisi Polri

## 📊 Klasifikasi Model (SVM + Grid Search)

| Informasi | Detail |
|---|---|
| **Teks Asli** | "{teks_ulasan}" |
| **Teks Setelah Preprocessing** | "{teks_bersih}" |
| **Prediksi Sentimen** | **{emoji_map[pred]}** |
| **Tingkat Keyakinan** | {confidence:.2f}% |

---

## 🤖 AI Insight (Powered by Groq - LLaMA 3.3 70B)

{insight}

---
*Analisis dilakukan oleh model SVM yang dioptimasi dengan Grid Search + Groq AI untuk interpretasi kontekstual.*
"""
    return hasil

print("✅ Fungsi prediksi + insight siap!")

✅ Fungsi prediksi + insight siap!


In [ ]:
# =====================================================================
# CELL 6: JALANKAN GRADIO DENGAN AI INSIGHT (GROQ) - VERSI FINAL
# =====================================================================
iface = gr.Interface(
    fn=prediksi_dengan_insight,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Masukkan ulasan pengguna Super App Presisi Polri di sini...\nContoh: 'Aplikasi bagus tapi sering error saat perpanjang SIM'",
        label="📝 Teks Ulasan (Input Manual)"
    ),
    outputs=gr.Markdown(label="📊 Hasil Analisis + AI Insight"),
    title="🚔 Sistem Analisis Sentimen Super App Presisi Polri",
    description="""
    **Arsitektur Sistem:**
    - **Model Klasifikasi:** Support Vector Machine (SVM) + Grid Search
    - **AI Interpretasi:** Groq Cloud (LLaMA 3.3 70B)
    - **Kelas Sentimen:** Positif | Netral | Negatif

    Sistem ini menggabungkan kekuatan SVM untuk klasifikasi cepat dengan kecerdasan LLaMA 3.3 via Groq untuk memberikan insight kontekstual dan rekomendasi bagi pengembang aplikasi.
    """,
    article="""
    ---
    **Dikembangkan oleh:** Carel Alberto Karma
    **Metode:** SVM + Grid Search + Groq AI (LLaMA 3.3)
    **Dataset:** Ulasan Google Play Store (2025–2026)
    **Package Name:** superapps.polri.presisi.presisi
    """,
    # Contoh ulasan untuk demo cepat (Positif, Negatif, dan Netral)
    examples=[
        ["Aplikasinya sangat bagus, perpanjangan SIM jadi lebih mudah dan cepat!"],
        ["Sering error dan lemot, tidak bisa dibuka sama sekali saat mau bayar STNK."],
        ["Bagaimana cara perpanjang STNK di aplikasi ini? Masih bingung menu nya."],
        ["Fitur pengaduan masyarakat sangat membantu, tapi loadingnya agak lama."],
        ["Aplikasi crash terus, sudah coba install ulang tetap sama. Kecewa banget!"],
        ["Lumayan bagus tapi kadang ngelag saat upload foto SKCK."],
        ["Apakah aplikasi ini bisa digunakan untuk perpanjangan SIM C? Mohon infonya."],
        ["Menu untuk cek tilang ETLE ada di mana ya? Tidak ketemu."]
    ]
)

print("🚀 Meluncurkan Gradio dengan Groq AI Insight...")
print("⏱️  Groq terkenal dengan kecepatan inferensi yang luar biasa!")
iface.launch(share=True, debug=True)

🚀 Meluncurkan Gradio dengan Groq AI Insight...
⏱️  Groq terkenal dengan kecepatan inferensi yang luar biasa!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aea2b2c9b4b35d631b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
